In [1]:
#include <iostream>
using namespace std;

# auto - automatic type inference

In [2]:
#include <vector>
void func_test_auto() {
    std::vector<int> v = {1, 2, 3, 4, 5};
    for(auto &x : v) {
        std::cout << x << " ";
    }
    std::cout << std::endl;
}

In [3]:
int main() {
    func_test_auto();
    return 0;
}
main();

1 2 3 4 5 


In [4]:
// output: 1 2 3 4 5 

# rvalue reference && , move semantics, std::move
lvalue: persistent obj
rvalue: temporary obj

In [5]:
#include <utility>
#include <string>
#include <iostream>
using namespace std;

std::string str1 = "Hello";
std::string str2 = std::move(str1); // Move str1 to str2
std::cout << "str1: " << str1 << std::endl; // str1 is now in a valid but unspecified state
std::cout << "str2: " << str2 << std::endl; // str2 contains the moved value

str1: 
str2: Hello


@0x7d1ea15fcc80

Move Semantic realization:
1. Move Constructor

In [6]:
class MyVector {
    int* data;
    size_t size;
public:
    MyVector(MyVector&& other) noexcept : data(other.data), size(other.size) {
        other.data = nullptr; // Leave the moved-from object in a valid state
        other.size = 0;
    }
    // Other members...
};

2. Move Assignment Operator

In [7]:
MyVector& operator=(MyVector&& other) noexcept {
    if (this != &other) {
        delete[] data; // Free existing resource
        data = other.data;
        size = other.size;
        other.data = nullptr; // Leave the moved-from object in a valid state
        other.size = 0;
    }
    return *this;
}

input_line_19:2:49: error: function definition is not allowed here
 MyVector& operator=(MyVector&& other) noexcept {
                                                ^


Interpreter Error: 

The 'std::unique_ptr' is the typical 'Only Move' type. Only allow to use std::move to transfer ownership.

# Smart Pointer: `std::shared_ptr` / `std::unique_ptr`
1. std::shared_ptr -> Reference Counting  
std::make_shared/new -> but have some differences: `new` create two memery blocks(store and count) separately, `make_shared` does that one-time.

In [ ]:
#include <iostream>
#include <vector>
#include <memory>
using namespace std;

std::shared_ptr<int> p1 = std::make_shared<int>(42);
std::shared_ptr<int> p2 = p1; // p2 now shares ownership of the integer
std::cout << "*p1: " << *p1 << std::endl;
std::cout << "*p2: " << *p2 << std::endl;
std::cout << "p1 use count: " << p1.use_count() << std::endl; // Outputs 2
p1.reset(); // p1 no longer owns the integer
std::cout << "p1 use count after reset: " << p1.use_count() << std::endl; // Outputs 0
std::cout << "p2 use count after p1 reset: " << p2.use_count() << std::endl; // Outputs 1
std::cout << "p2 value: " << p2.get() << std::endl; // Outputs the address of the integer


Attention:
- Circular Reference

In [1]:
#include <iostream>
#include <memory>
#include <string>

struct Apartment;

struct Person {
    std::string name;
    std::shared_ptr<Apartment> apartment; // Shared ownership of Apartment
    Person(const std::string& n) : name(n) { std::cout << "Person " << name << " created.\n"; }
    ~Person() { std::cout << "Person " << name << " destroyed.\n"; }
};

struct Apartment {
    std::string address;
    std::shared_ptr<Person> tenant; // Shared ownership of Person
    Apartment(const std::string& addr) : address(addr) { std::cout << "Apartment at " << address << " created.\n"; }
    ~Apartment() { std::cout << "Apartment at " << address << " destroyed.\n"; }
};

{
    std::shared_ptr<Person> john = std::make_shared<Person>("John");  // John counts as 1
    std::shared_ptr<Apartment> apt = std::make_shared<Apartment>("123 Main St");  // Apartment counts as 1

    john->apartment = apt; // John owns the apartment, increasing apt's use count to 2
    apt->tenant = john;    // Apartment owns John, increasing john's use count to 2

    // At this point, both john and apt have a use count of 2
    std::cout << "John's use count: " << john.use_count() << std::endl; // Outputs 2
    std::cout << "Apartment's use count: " << apt.use_count() << std::endl; // Outputs 2
} // Both john and apt go out of scope here, but they won't be destroyed due to circular reference

std::cout << "End of main function.\n";

// To break the circular reference, we can use weak_ptr instead of shared_ptr for one of the relationships.



Person John created.
Apartment at 123 Main St created.
John's use count: 2
Apartment's use count: 2
End of main function.


@0x7860793fcc80

Solution -> std::weak_ptr  
Strong Ownership  
Weak Reference 

The steps or process to use std::weak_ptr:  
1. call lock() to get a valid std::shared_ptr or nullptr
2. use expired() to check the object if released

In [ ]:
#include <iostream>
#include <memory>
#include <string>

struct Apartment;

struct Person {
    std::string name;
    std::weak_ptr<Apartment> apartment; // Weak reference to Apartment
    Person(const std::string& n) : name(n) { std::cout << "Person " << name << " created.\n"; }
    ~Person() { std::cout << "Person " << name << " destroyed.\n"; }
};

struct Apartment {
    std::string address;
    std::weak_ptr<Person> tenant; // Weak reference to Person
    Apartment(const std::string& addr) : address(addr) { std::cout << "Apartment at " << address << " created.\n"; }
    ~Apartment() { std::cout << "Apartment at " << address << " destroyed.\n"; }
};

{
    std::shared_ptr<Person> john = std::make_shared<Person>("John");  // John counts as 1
    std::shared_ptr<Apartment> apt = std::make_shared<Apartment>("123 Main St");  // Apartment counts as 1

    john->apartment = apt; // weak_ptr does not increase the use count of apt
    apt->tenant = john;    // weak_ptr does not increase the use count of john

    // At this point, both john and apt have a use count of 1
    std::cout << "John's use count: " << john.use_count() << std::endl; // Outputs 1
    std::cout << "Apartment's use count: " << apt.use_count() << std::endl; // Outputs 1

    // Lock the weak_ptr to obtain a shared_ptr(atomic and weak count+1) before using the object.
    // Do not check expired() first: another thread could destroy the object
    // between that check and lock(). lock() safely returns an empty shared_ptr
    // if the object is no longer available.
    if (auto aptPtr = john->apartment.lock()) { // Lock the weak_ptr to get a shared_ptr
        std::cout << "John lives at: " << aptPtr->address << std::endl;
    } else {
        std::cout << "John's apartment is no longer available.\n";
    }
} // Both john and apt go out of scope here, but they won't be destroyed due to circular reference

std::cout << "End of main function.\n";



Person John created.
Apartment at 123 Main St created.
John's use count: 1
Apartment's use count: 1
John lives at: 123 Main St
Apartment at 123 Main St destroyed.
Person John destroyed.
End of main function.


@0x734bd1dfcc80

2. std::unique_ptr (Exclusive Ownership Smart Pointer)  
- Zero-cost Abstraction (8bits tuple, same as Raw Pointer)
- Exclusive Ownership (Copy)  
The `std::make_unique` can only be used from C++14  
Use Cases:  
- Factory Pattern


# Lambda Expression  
[ 擷取列表 ] ( 參數列表 ) 屬性/修飾符 -> 回傳型態 { 函式本體 }  
// 例如：
auto func = [factor](int a) mutable -> int { return a * factor; };

# std::function and std::bind
std::function 和 std::bind 是用于实现函数对象封装和参数绑定的核心工具。它们极大地增强了 C++ 处理回调函数、事件驱动编程以及函数式编程的能力。  

1. std::function（函数包装器）  

std::function 是一个通用的、多态的函数封装器。它可以存储、复制和调用任何可以调用的目标（如普通函数、Lambda 表达式、函数对象/仿函数、以及类成员函数）。
• 语法： std::function<返回值类型(参数类型1, 参数类型2, ...)>

In [1]:
#include <iostream>
#include <functional>

void printNum(int i) {
    std::cout << "Normal function: " << i << std::endl;
}

struct Foo {
    void operator()(int i) const {
        std::cout << "Functor: " << i << std::endl;
    }
};

// main
// 1. 封装普通函数
std::function<void(int)> f_normal = printNum;
f_normal(42);

// 2. 封装 Lambda 表达式
std::function<void(int)> f_lambda = [](int i) {
    std::cout << "Lambda: " << i << std::endl;
};
f_lambda(42);

// 3. 封装仿函数（函数对象）
std::function<void(int)> f_functor = Foo();
f_functor(42);

return 0;


Normal function: 42
Lambda: 42
Functor: 42



2. std::bind（参数绑定器）

std::bind 是一个参数绑定器。它接收一个可调用对象，并生成一个新的可调用对象。它可以用来改变原函数的参数顺序、固定某些参数，或者将类成员函数与具体的对象实例进行绑定。
• 占位符： std::placeholders::_1, _2, _3 等，代表新生成的可调用对象在调用时需要传入的第 1、2、3 个参数。

In [2]:
#include <iostream>
#include <functional>

void output(int x, int y, int z) {
    std::cout << "x=" << x << ", y=" << y << ", z=" << z << std::endl;
}

// int main() {
    using namespace std::placeholders;

    // 固定第一个参数 x = 10，其余参数由新函数传入
    // _1 对应新函数的第一个参数（传给 y），_2 对应新函数的第二个参数（传给 z）
    auto bindFunc1 = std::bind(output, 10, _1, _2);
    bindFunc1(20, 30); // 输出: x=10, y=20, z=30

    // 改变参数顺序
    // _2 对应新函数的第二个参数（传给 x），_1 对应新函数的第一个参数（传给 y）
    auto bindFunc2 = std::bind(output, _2, _1, 100);
    bindFunc2(10, 20); // 10传给_1, 20传给_2 -> 输出: x=20, y=10, z=100

//     return 0;
// }


x=10, y=20, z=30
x=20, y=10, z=100


In [ ]:

3. std::function 与 std::bind 结合（绑定类成员函数）

在 C++ 中，调用类的非静态成员函数必须要有一个具体的对象实例。`std::bind 最常用的场景就是将类的成员函数与其对象指针/引用绑定，然后赋值给 std::function。`

In [3]:
#include <iostream>
#include <functional>

class Parser {
public:
    void parse(int code, const std::string& msg) {
        std::cout << "Parsing code: " << code << ", msg: " << msg << std::endl;
    }
};

// int main() {
    Parser my_parser;

    // 成员函数需要隐式传入 `this` 指针，所以 bind 的第一个参数必须是对象地址或引用
    // _1 和 _2 对应新包装函数的两个参数
    std::function<void(int, const std::string&)> callback = 
        std::bind(&Parser::parse, &my_parser, std::placeholders::_1, std::placeholders::_2);

    // 像触发普通函数一样触发类成员函数
    callback(200, "OK"); // 输出: Parsing code: 200, msg: OK

//     return 0;
// }


Parsing code: 200, msg: OK



4. 现代 C++ 最佳实践：std::bind vs Lambda

随着 C++11/14/17 的普及，现代 C++ 强烈推荐使用 Lambda 表达式来替代 std::bind。

特性	        std::bind	                        Lambda 表达式 (推荐)
可读性	        占位符 _1, _2 容易让人混乱	            直观的代码块，易读易懂
性能	        往往带来更多的编译期开销和运行时包装	    编译器能更好地进行内联优化 (Inline)
参数类型检查	  报错信息极其冗长、难懂	                报错信息更直接、清晰


In [ ]:
// 使用 std::bind 绑定成员函数
auto f_bind = std::bind(&Parser::parse, &my_parser, std::placeholders::_1, "Fixed Message");
f_bind(404);

// 使用 Lambda 替代（更现代、更高效）
auto f_lambda = [&my_parser](int code) { 
    my_parser.parse(code, "Fixed Message"); 
};
f_lambda(404);
